## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <bits/stdc++.h>
using namespace std;

namespace {
    constexpr int MAX_N = 1024 + 5;

    int n;
    int swapValueA, swapValueB;
    int groupBlockSize;

    int permutation[MAX_N];
    int valuePosition[MAX_N];
    vector<int> operations;

    inline int readInt() {
        int value = 0;
        char ch = getchar();
        while (ch < '0' || ch > '9') {
            ch = getchar();
        }
        while (ch >= '0' && ch <= '9') {
            value = value * 10 + (ch - '0');
            ch = getchar();
        }
        return value;
    }

    inline void rebuildValuePosition() {
        for (int i = 0; i < n; ++i) {
            valuePosition[permutation[i]] = i;
        }
    }

    inline void applySwapMagic() {
        operations.push_back(0);
        for (int i = 0; i < n; ++i) {
            if (permutation[i] == swapValueA) {
                permutation[i] = swapValueB;
            } else if (permutation[i] == swapValueB) {
                permutation[i] = swapValueA;
            }
        }
        rebuildValuePosition();
    }

    inline void applyAddMagic(int delta) {
        delta %= n;
        if (delta < 0) {
            delta += n;
        }
        if (delta == 0) {
            return;
        }

        operations.push_back(delta);
        for (int i = 0; i < n; ++i) {
            permutation[i] = (permutation[i] + delta) % n;
        }
        rebuildValuePosition();
    }

    inline void applyXorMagic(int mask) {
        if (mask == 0) {
            return;
        }

        operations.push_back(-mask);
        for (int i = 0; i < n; ++i) {
            permutation[i] ^= mask;
        }
        rebuildValuePosition();
    }

    inline void getPairMappedPosition(int x, int y, int &mappedX, int &mappedY) {
        int delta = (y - x + n - groupBlockSize + n) % n;
        mappedX = 0;
        mappedY = 0;

        for (int step = n / 2; step >= 2 * groupBlockSize; step >>= 1) {
            if (delta >= step) {
                delta -= step;
                mappedY += step / 2;
            } else {
                mappedX += step / 2;
            }
        }

        mappedX += n / 2;
        mappedX += (x & (groupBlockSize - 1));
        mappedY += (x & (groupBlockSize - 1));
    }

    void applySwapBetweenValues(int x, int y) {
        if ((x / groupBlockSize) % 2 == (y / groupBlockSize) % 2) {
            int middleValue;
            if ((x / groupBlockSize) % 2 == 0) {
                middleValue = (x & (groupBlockSize - 1)) + groupBlockSize;
            } else {
                middleValue = (x & (groupBlockSize - 1));
            }

            applySwapBetweenValues(x, middleValue);
            applySwapBetweenValues(y, middleValue);
            applySwapBetweenValues(x, middleValue);
            return;
        }

        int mappedA, mappedB, mappedX, mappedY;
        getPairMappedPosition(swapValueA, swapValueB, mappedA, mappedB);
        getPairMappedPosition(x, y, mappedX, mappedY);

        applyAddMagic((mappedX - x + n) % n);
        applyXorMagic(mappedX ^ mappedA);
        applyAddMagic((swapValueA - mappedA + n) % n);

        applySwapMagic();

        applyAddMagic((mappedA - swapValueA + n) % n);
        applyXorMagic(mappedX ^ mappedA);
        applyAddMagic((x - mappedX + n) % n);
    }

    struct ReducedPermutation {
        int value[MAX_N];
        int length;
        vector<int> ops;

        bool operator<(const ReducedPermutation &other) const {
            for (int i = 0; i < length; ++i) {
                if (value[i] != other.value[i]) {
                    return value[i] < other.value[i];
                }
            }
            return false;
        }

        bool isSorted() const {
            for (int i = 0; i + 1 < length; ++i) {
                if (value[i] > value[i + 1]) {
                    return false;
                }
            }
            return true;
        }

        ReducedPermutation inverse() const {
            ReducedPermutation result;
            result.length = length;
            for (int i = 0; i < length; ++i) {
                result.value[value[i]] = i;
            }
            return result;
        }

        bool build() {
            bool visited[100005] = {};
            for (int i = 0; i < length; ++i) {
                visited[i] = true;
            }
            for (int i = 0; i < length; ++i) {
                if (!visited[i]) {
                    return false;
                }
            }

            if (length == 1) {
                return true;
            }

            ReducedPermutation leftHalf, rightHalf;
            leftHalf.length = rightHalf.length = length / 2;

            for (int i = 0; i < length / 2; ++i) {
                leftHalf.value[i] = value[i * 2] / 2;
                rightHalf.value[i] = value[i * 2 + 1] / 2;
            }

            if (!leftHalf.build() || !rightHalf.build()) {
                return false;
            }

            if (value[0] & 1) {
                ops.push_back(length == 2 ? 1 : -1);
            }

            int leftXorAccumulation = 0;
            for (int op : leftHalf.ops) {
                if (op > 0) {
                    ops.push_back(-1);
                    ops.push_back(1);
                } else {
                    ops.push_back(op * 2);
                    leftXorAccumulation ^= (-op) * 2;
                }
            }
            if (leftXorAccumulation) {
                ops.push_back(-leftXorAccumulation);
            }

            int rightXorAccumulation = 0;
            for (int op : rightHalf.ops) {
                if (op > 0) {
                    ops.push_back(1);
                    ops.push_back(-1);
                } else {
                    ops.push_back(op * 2);
                    rightXorAccumulation ^= (-op) * 2;
                }
            }

            if ((rightXorAccumulation & (length / 2)) != (leftXorAccumulation & (length / 2))) {
                return false;
            }

            if (leftXorAccumulation >= length / 2) {
                leftXorAccumulation -= length / 2;
            }
            if (rightXorAccumulation >= length / 2) {
                rightXorAccumulation -= length / 2;
            }
            if (leftXorAccumulation != rightXorAccumulation) {
                return false;
            }

            vector<int> mergedOps;
            for (int op : ops) {
                if (mergedOps.empty()) {
                    mergedOps.push_back(op);
                } else {
                    if (op < 0 && mergedOps.back() < 0) {
                        mergedOps.back() = -((-mergedOps.back()) ^ (-op));
                        if (mergedOps.back() == 0) {
                            mergedOps.pop_back();
                        }
                    } else {
                        mergedOps.push_back(op);
                    }
                }
            }
            ops.swap(mergedOps);
            return true;
        }
    };
}

int main() {
    n = readInt();
    swapValueA = readInt();
    swapValueB = readInt();

    for (int i = 0; i < n; ++i) {
        permutation[i] = readInt();
    }

    rebuildValuePosition();

    groupBlockSize = (swapValueA - swapValueB + n) % n;
    groupBlockSize &= -groupBlockSize;
    if (groupBlockSize == 0) {
        groupBlockSize = n;
    }

    if (groupBlockSize > 1) {
        ReducedPermutation basePermutation;
        basePermutation.length = groupBlockSize;
        for (int i = 0; i < n; ++i) {
            basePermutation.value[i] = permutation[i] & (groupBlockSize - 1);
        }

        if (!basePermutation.build()) {
            printf("-1\n");
            return 0;
        }

        for (int op : basePermutation.ops) {
            if (op > 0) {
                applyAddMagic(op);
            } else {
                applyXorMagic(-op);
            }
        }
    }

    for (int remainder = 0; remainder < groupBlockSize; ++remainder) {
        vector<int> bucketValues;
        for (int j = remainder; j < n; j += groupBlockSize) {
            bucketValues.push_back(permutation[j]);
        }

        sort(bucketValues.begin(), bucketValues.end());

        bool valid = true;
        int pointer = 0;
        for (int j = remainder; j < n; j += groupBlockSize) {
            if (bucketValues[pointer] != j) {
                valid = false;
                break;
            }
            ++pointer;
        }

        if (!valid) {
            printf("-1\n");
            return 0;
        }

        for (int j = remainder; j < n; j += groupBlockSize) {
            if (permutation[j] != j) {
                applySwapBetweenValues(j, permutation[j]);
            }
        }
    }

    for (int i = 0; i < n; ++i) {
        assert(permutation[i] == i);
    }

    printf("%d\n", static_cast<int>(operations.size()));
    for (int op : operations) {
        if (op == 0) {
            printf("0\n");
        } else if (op < 0) {
            printf("1 %d\n", -op);
        } else {
            printf("2 %d\n", op);
        }
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct {
    int p;  // 位置
    int c;  // 花费
} Shop;

typedef struct {
    int now;  // 当前位置
    int s;    // 剩余的钱
    int id;   // 已经考虑到的商店索引
} Node;

typedef struct {
    Node data[100005];
    int front;
    int rear;
} Queue;

void initQueue(Queue *q) {
    q->front = 0;
    q->rear = 0;
}

int isEmpty(Queue *q) {
    return q->front == q->rear;
}

void push(Queue *q, Node node) {
    q->data[q->rear++] = node;
}

Node pop(Queue *q) {
    return q->data[q->front++];
}

int cmp(const void *a, const void *b) {
    Shop *s1 = (Shop *)a;
    Shop *s2 = (Shop *)b;
    if (s1->p != s2->p)
        return s1->p - s2->p;
    return s1->c - s2->c;
}

int main() {
    int n, l, ma, s;
    
    while (scanf("%d %d %d %d", &n, &l, &ma, &s) != EOF) {
        Shop a[2005];
        
        for (int i = 1; i <= n; i++) {
            scanf("%d %d", &a[i].p, &a[i].c);
        }
        
        // 如果直接能到达终点
        if (ma >= l) {
            printf("Yes\n");
            continue;
        }
        
        // 添加终点作为一个特殊的"商店"
        a[++n].p = l;
        a[n].c = 0;
        
        // 排序
        qsort(a + 1, n, sizeof(Shop), cmp);
        
        // BFS
        Queue q;
        initQueue(&q);
        Node st = {0, s, 0};
        push(&q, st);
        
        int ok = 0;
        
        while (!isEmpty(&q)) {
            Node x = pop(&q);
            
            if (x.now == l) {
                ok = 1;
                break;
            }
            
            // 尝试到达所有在体力范围内的商店
            for (int i = x.id + 1; i <= n; i++) {
                if (a[i].p - x.now > ma) {
                    // 超出体力范围
                    break;
                }
                
                // 如果有足够的钱在这个商店补给
                if (x.s >= a[i].c) {
                    Node t = {a[i].p, x.s - a[i].c, i};
                    push(&q, t);
                }
            }
        }
        
        printf("%s\n", ok ? "Yes" : "No");
    }
    
    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define maxn 2000050

char a[maxn], newa[maxn];
int p1[maxn];

char b[maxn], newb[maxn];
int p2[maxn];

// 手写 max 和 min
int max(int x, int y) {
    return x > y ? x : y;
}

int min(int x, int y) {
    return x < y ? x : y;
}

int init1() {
    int len = strlen(a);
    newa[0] = '$';
    newa[1] = '#';
    int j = 2;
    for (int i = 0; i < len; i++) {
        newa[j++] = a[i];
        newa[j++] = '#';
    }
    newa[j] = '\0';
    return j;
}

int init2() {
    int len = strlen(b);
    newb[0] = '$';
    newb[1] = '#';
    int j = 2;
    for (int i = 0; i < len; i++) {
        newb[j++] = b[i];
        newb[j++] = '#';
    }
    newb[j] = '\0';
    return j;
}

void manacher1() {
    int len = init1();
    int id = 0;
    int mx = 0;
    for (int i = 1; i < len; i++) {
        if (i < mx)
            p1[i] = min(p1[2 * id - i], mx - i);
        else
            p1[i] = 1;

        while (newa[i - p1[i]] == newa[i + p1[i]])
            p1[i]++;

        if (mx < i + p1[i]) {
            id = i;
            mx = i + p1[i];
        }
    }
}

void manacher2() {
    int len = init2();
    int id = 0;
    int mx = 0;
    for (int i = 1; i < len; i++) {
        if (i < mx)
            p2[i] = min(p2[2 * id - i], mx - i);
        else
            p2[i] = 1;

        while (newb[i - p2[i]] == newb[i + p2[i]])
            p2[i]++;

        if (mx < i + p2[i]) {
            id = i;
            mx = i + p2[i];
        }
    }
}

int main() {
    int n;
    scanf("%d", &n);
    scanf("%s", a);
    scanf("%s", b);

    manacher1();
    manacher2();

    int ans = 1;
    n = (n + 1) * 2;

    for (int i = 2; i <= n; i++) {  // 匹配 '#'
        int tmp = max(p1[i], p2[i - 2]);
        while (newa[i - tmp] == newb[i + tmp - 2])
            tmp++;
        ans = max(ans, tmp);
    }

    printf("%d\n", ans - 1);
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define MAXN 555555

char c[MAXN];
int id[MAXN];

// 模拟 map
int a[MAXN];

// 模拟 set（有序数组）
int pos[MAXN];
int pos_size;

// 二分查找 >= x 的第一个位置
int lower_bound(int x) {
    int l = 0, r = pos_size - 1, ans = pos_size;
    while (l <= r) {
        int mid = (l + r) / 2;
        if (pos[mid] >= x) {
            ans = mid;
            r = mid - 1;
        } else {
            l = mid + 1;
        }
    }
    return ans;
}

// 插入（保持有序）
void insert_pos(int x) {
    int i = pos_size - 1;
    while (i >= 0 && pos[i] > x) {
        pos[i + 1] = pos[i];
        i--;
    }
    pos[i + 1] = x;
    pos_size++;
}

// 删除第 idx 个
void erase_pos(int idx) {
    for (int i = idx; i < pos_size - 1; i++) {
        pos[i] = pos[i + 1];
    }
    pos_size--;
}

int main() {
    int n;
    while (scanf("%d", &n) != EOF) {

        int ans = -1;
        int flag = 1;

        pos_size = 0;
        memset(a, 0, sizeof(a));

        char g[2];

        for (int i = 1; i <= n; i++) {
            scanf("%s", g);
            c[i] = g[0];

            if (c[i] != '?') {
                scanf("%d", &id[i]);
            }

            if (flag == 0) continue;

            if (c[i] == '?') {
                insert_pos(i);
                continue;
            }

            if (a[id[i]] != 0) {
                if (c[a[id[i]]] == c[i]) {
                    int idx = lower_bound(a[id[i]]);
                    if (idx == pos_size) {
                        ans = i;
                        flag = 0;
                    } else {
                        erase_pos(idx);
                    }
                }
                a[id[i]] = i;
            } else {
                if (c[i] == 'O') {
                    int idx = lower_bound(0);
                    if (idx == pos_size) {
                        ans = i;
                        flag = 0;
                    } else {
                        erase_pos(idx);
                    }
                }
                a[id[i]] = i;
            }
        }

        printf("%d\n", ans);
    }

    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include<iostream>
#include<cstdio>
#include<algorithm>
using namespace std;
int father[1001];

int find(int x)
{
    if(father[x]==x)
       return x;
    else
       {
           father[x]=find(father[x]);
           return  father[x];
       }
}

void merge(int a,int b)
{
    int fa=find(a);
    int fb=find(b);
    if(fa!=fb)
    {
      father[fb]=fa;
    }
}

int main()
{
    int n;
    while(cin>>n&&n)
    {
        int x[n+1],y[n+1],ans=0;

        for(int i=1;i<=n;i++)
            father[i]=i;
        for(int i=1;i<=n;i++)
        {
            cin>>x[i]>>y[i];
        }

        for(int i=1;i<n;i++)
        {
            for(int j=i+1;j<=n;j++)
            {
               if(x[i]==x[j]||y[i]==y[j])
                {
                    merge(i,j);
                }
            }
        }

        for(int i=1;i<=n;i++)
        {
           if(father[i]==i)
               ans++;
        }
        cout<<ans-1<<endl;
    }
     return 0;
}



## F 通配符匹配

In [ ]:
## add your code here
#include<iostream>
#include<cstdio>
#include<algorithm>
#include<cstring>
#include<cmath>
 
using namespace std;
 
typedef unsigned long long ull;
int t, num = 1, trush[114514];
ull base[114514 << 1], ask[114514];
string temp, common;
 
struct Wildcard_String {
	ull simple_len[114514];
	ull simple[114514];
}wild;
 
void PreTreatMent() {
	int len = 0;
	base[0] = 1;
	for (int i = 1; i <= 114514; ++ i) 
		base[i] = base[i - 1] * 1331;
	for (int i = 0; i < temp.size(); ++ i) {
		if (temp[i] == '*' || temp[i] == '?') {
			wild.simple_len[i - len] = len;
			wild.simple[i - len] = wild.simple[i - 1];
			//将预处理完的状态都转移到每个非通配符串的第一个 
			len = 0;
		}
		else len ++, wild.simple[i] = wild.simple[i - 1] * base[1] + (ull)temp[i];
	}
	wild.simple_len[temp.size() - len] = len;
	wild.simple[temp.size() - len] = wild.simple[temp.size() - 1];
	return;
} 
 
bool Work(int get_pos, int ask_pos) {
	if (get_pos >= temp.size()) { //当通配符串到最后一位 
		if (ask_pos >= common.size() || temp[temp.size() - 1] == '*') return true;
		//若匹配串也为最后一位或通配符串最后一位为"*" 
		else return false;//若匹配串不为最后一位 
	}
	if (ask_pos == common.size()) {
		if (get_pos == temp.size() - 1 && temp[temp.size() - 1] == '*') return true;
		//注意特判通配符串最后一个是否为*，因为*可以匹配0个字符 
		else return false;
	}
	//此时代表通配符串不为最后一位但匹配串在最后一位 
	if (temp[get_pos] == '?') {
		return Work(get_pos + 1, ask_pos + 1);
		//若为通配符"?"，则两个字符串不用管get_pos位置的字符 
	}
	if (temp[get_pos] == '*') {
		//若get_pos位置为通配符"*" 
		for (int i = ask_pos; i < common.size(); ++ i) {
			//将被匹配字符串一一向后匹配 
			if (i >= common.size()) return false;
			if (Work(get_pos + 1, i)) return true;
		}
		return false;
	}
	//若get_pos位置为普通字符则直接比较
	if (ask_pos + wild.simple_len[get_pos] - 1 >= common.size()) {
		return false;
	}
	//若匹配字符串剩下的长度不够 
	int len = wild.simple_len[get_pos];
	int al = ask_pos, ar = al + len - 1;
	ull gans = wild.simple[get_pos], aans = ask[ar] - ask[al - 1] * base[ar - al + 1];
	//将两个子串哈希值处理出来比较 
	if (gans == aans) return Work(get_pos + len, ar + 1);
	else return false;
}
 
int main() {
	cin >> temp >> t;
	PreTreatMent();
	while (t --) {
		cin >> common;
		for (int i = 0; i < common.size(); ++ i) {
			ask[i] = ask[i - 1] * base[1] + (ull)common[i];
		}
		if (Work(0, 0)) cout << "YES" << endl;
		else cout << "NO" << endl; 
	}
	return 0; 
} 

## G 汉诺塔

In [ ]:
## add your code here
#include<bits/stdc++.h>
using namespace std;
#define ll long long
const ll N=302;
ll n;
bool vis[N];
ll f[4][N],g[4][N];
//f(x,i):从x移走1~i的步数
//g(x,i):从x移走1~i移到了哪里 
int main()
{
	cin>>n;
	for(int i=1;i<=6;i++)
	{
		string s;
		cin>>s;
		ll x=s[0]-'A'+1,y=s[1]-'A'+1;
		if(vis[x])continue;
		f[x][1]=1,g[x][1]=y;
		vis[x]=1;
	}
	for(int i=2;i<=n;i++)
	{
		for(int x=1;x<=3;x++)
		{
			ll y=g[x][i-1],z=6-x-y;
			if(g[y][i-1]==z)
			{
				g[x][i]=z;
				f[x][i]=f[x][i-1]+1+f[y][i-1];
				//1~i-1 x->y
				//i x->z
				//1~i-1 x->z
			}
			if(g[y][i-1]==x)
			{
				g[x][i]=y;
				f[x][i]=f[x][i-1]+1+f[y][i-1]+1+f[x][i-1];
				//1~i-1 x->y
				//i y->z
				//1~i-1 y->x
				//i z->y
				//1~i-1 x->y
			}
		}
	}
	cout<<f[1][n];
	return 0;
}


## H 马步距离

In [ ]:
## add your code here
#include<cstdio>
#include<algorithm>
#include<cstring>
#include<cmath>
#include<queue>
using namespace std;

#define N 205

int sx,sy,ex,ey;
int dx[8]={1,-1,1,-1,2,-2,2,-2};
int dy[8]={-2,-2,2,2,-1,1,1,-1};

int dis[N][N],vis[N][N];
int x,y;

void bfs()
{
    queue<pair<int,int> > q;
    q.push(make_pair(x,y));
    vis[x][y]=1;

    while(!q.empty())
    {
        pair<int,int> P=q.front();
        q.pop();

        if(P.first==50 && P.second==50) return;

        for(int i=0;i<8;i++)
        {
            int nx=P.first+dx[i];
            int ny=P.second+dy[i];

            if(nx<0 || ny<0 || nx>=N || ny>=N) continue;
            if(vis[nx][ny]) continue;

            vis[nx][ny]=1;
            dis[nx][ny]=dis[P.first][P.second]+1;
            q.push(make_pair(nx,ny));
        } 
    }
}

int main()
{
    scanf("%d%d%d%d",&sx,&sy,&ex,&ey);

    x=abs(sx-ex);
    y=abs(sy-ey);

    int ans=0;

    while(x+y>50)
    {
        if(x<y) swap(x,y);

        if(x-4>=2*y)
            x-=4,ans+=2;
        else
            x-=2,y-=1,ans++;
    } 

    x+=50;
    y+=50;

    bfs();

    printf("%d\n",ans+dis[50][50]);
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
/**
 *
 * 
 * @param heights int整型一维数组 
 * @param heightsLen int heights数组长度
 * @return int整型
 *
 */
int largestRectangleArea(int* heights, int heightsLen ) {
    int stack[heightsLen + 1];  // 单调栈，存下标
    int top = -1;
    int maxArea = 0;

    for (int i = 0; i <= heightsLen; i++) {
        int curHeight = (i == heightsLen) ? 0 : heights[i];

        // 当前高度小于栈顶高度，开始计算面积
        while (top != -1 && curHeight < heights[stack[top]]) {
            int h = heights[stack[top--]];

            int width;
            if (top == -1) {
                width = i;
            } else {
                width = i - stack[top] - 1;
            }

            int area = h * width;
            if (area > maxArea) {
                maxArea = area;
            }
        }

        stack[++top] = i;
    }

    return maxArea;
}

## J 消防局的设立

In [ ]:
## add your code here
#include <iostream>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <algorithm>
#include <cmath>
#include <cctype>
#include <vector>

using namespace std;

inline int gi()
{
	int f = 1, x = 0;
	char c = getchar();
	while (c < '0' || c > '9')
	{
		if (c == '-') f = -1;
		c = getchar();
	}
	while (c >= '0' && c <= '9')
	{
		x = x * 10 + c - '0';
		c = getchar();
	}
	return f * x;
}

int n, m, mm, dis[1003], ans, u, v, fa[1003], deep[1003], r[1003];
vector <int> w[1003];

int main()
{
	n = gi();
	fa[1] = 1;
	for (int i = 2; i <= n; i++)
	{
		int x = gi();
		fa[i] = x, dis[i] = dis[x] + 1, w[x].push_back(i);
	}
	for (int i = 1; i <= n; i++)
	{
		int k = 0, mk = -1;
		for (int j = 1; j <= n; j++)
		{
			if (!r[j] && dis[j] > mk) k = j, mk = dis[j];
		}
		if (k == 0)
		{
			printf("%d\n", i - 1);
			return 0;
		}
		m = fa[fa[k]], r[m] = 1, r[fa[m]] = 1, r[fa[fa[m]]] = 1;
		for (int j = 0; j < w[m].size(); j++)
		{
			u = w[m][j], r[u] = 1;
			for (int l = 0; l < w[u].size(); l++)
			{
				r[w[u][l]] = 1;
			}
		}
		mm = fa[m];
		for (int j = 0; j < w[mm].size(); j++)
		{
			r[w[mm][j]] = 1;
		}
	}
	return 0;
}